# 5. EDA with SQL (SQLite)
Run the data-wrangling notebook first. This notebook writes the processed table to SQLite and executes SQL questions relevant to the capstone.

In [ ]:
import pandas as pd, sqlite3
from pathlib import Path

ROOT = Path("..")
df = pd.read_csv(ROOT / "data" / "processed" / "spacex_falcon9_launches.csv")
DB = ROOT / "data" / "processed" / "spacex_capstone.sqlite"
con = sqlite3.connect(DB)
df.to_sql("launches", con, if_exists="replace", index=False)
print("Database:", DB, "| rows:", len(df))
display(pd.read_sql_query("SELECT * FROM launches LIMIT 5", con))

In [ ]:
queries = {
"launch_counts_by_site": """SELECT launch_site, COUNT(*) AS launch_count
FROM launches GROUP BY launch_site ORDER BY launch_count DESC""",
"payload_statistics": """SELECT MIN(payload_mass_kg) AS min_payload_kg,
AVG(payload_mass_kg) AS avg_payload_kg, MAX(payload_mass_kg) AS max_payload_kg
FROM launches WHERE payload_mass_kg IS NOT NULL""",
"mission_success_counts": """SELECT mission_success, COUNT(*) AS records
FROM launches GROUP BY mission_success ORDER BY records DESC""",
"yearly_launch_counts": """SELECT year, COUNT(*) AS launch_count
FROM launches WHERE year IS NOT NULL GROUP BY year ORDER BY year""",
"landing_attempt_results": """SELECT landing_attempt, landing_success, COUNT(*) AS records
FROM launches GROUP BY landing_attempt, landing_success ORDER BY records DESC""",
"top_sites_by_landing_success": """SELECT launch_site, COUNT(*) AS labelled_attempts,
ROUND(100.0 * AVG(CAST(landing_success AS REAL)), 2) AS success_rate_pct
FROM launches WHERE landing_attempt = 1 AND landing_success IS NOT NULL
GROUP BY launch_site HAVING COUNT(*) >= 3 ORDER BY success_rate_pct DESC"""
}
for name, query in queries.items():
    print("\n", name.upper())
    result = pd.read_sql_query(query, con)
    display(result)
    result.to_csv(ROOT / "outputs" / "model_results" / f"sql_{name}.csv", index=False)
con.close()

## Note
The API dataset differs from the legacy IBM course dataset and the query outputs can change as new launches are added. Include the query text and the generated result tables in your final slides. Do not present the query list alone as completed analysis.